In [45]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [46]:
import numpy as np

In [47]:
training_sample = "I deposited my money in the bank by the river bank"

In [48]:
sequence = training_sample.lower().split(" ")

vocab = list(set(sequence))

vocab.sort()

idx_to_word = { i: word for i,word in enumerate(vocab)}
word_to_idx = { word: i for i, word in enumerate(vocab)}

vocab_size = len(vocab)

In [49]:
one_hot_sequence = np.zeros((len(sequence), vocab_size), dtype=int)

for step, word in enumerate(sequence):
    word_idx = word_to_idx[word]
    one_hot_sequence[step, word_idx] = 1

In [50]:
one_hot_sequence

array([[0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1],
       [1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 0]])

In [51]:
class RNN():
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.learning_rate = learning_rate
        self.U = np.random.randn(hidden_size, input_size) * 0.1
        self.W = np.random.randn(hidden_size, hidden_size) * 0.1
        self.V = np.random.randn(output_size, hidden_size) * 0.1
        self.b_h = np.zeros((hidden_size, 1))
        self.b_y = np.zeros((output_size, 1))
        self.gradients = []

    def loss(self, y_t, y_pred):
        prob_k = np.clip(np.sum(y_t * y_pred), 1e-15, 1.0)
        return -np.log(prob_k)

    def softmax(self, x):
        exp_x = np.exp(x - np.max(x))
        return exp_x / np.sum(exp_x)

    def initialize_hidden(self):
        return np.zeros((self.hidden_size, 1))
    
    def forward(self, x_t, h_prev):
        a_t = np.dot(self.U, x_t) + np.dot(self.W, h_prev) + self.b_h
        h_t = np.tanh(a_t)
        return a_t, h_t

    def save_gradients(self, dU, dW, dV, db_h, db_y):
        self.gradients.append((dU, dW, dV, db_h, db_y))

    def get_gradients(self):
        return self.gradients
    
    def backpropagate_through_time(self, sequence):
        T = len(sequence)
        h_prev = self.initialize_hidden()
        h_states = [h_prev]
        a_states = []
        for x_t in sequence[:-1]: 
            x_t = x_t.reshape(-1,1)
            a_t, h_prev = self.forward(x_t, h_prev)
            h_states.append(h_prev)
            a_states.append(a_t)

        h_T = h_prev
        o_T = np.dot(self.V, h_T) + self.b_y
        y_pred = self.softmax(o_T)
        y_true = sequence[-1].reshape(-1,1)
        loss = self.loss(y_true, y_pred)


        # Initialize gradients
        dU = np.zeros_like(self.U)
        dW = np.zeros_like(self.W)

        db_h = np.zeros_like(self.b_h)

        dL_by_doT = y_pred - y_true # Derrivate of softmax and cross-entropy combined

        # dL_by_dV  = dL_by_doT * (doT / dV)T
        # doT / dV = h_T_transpose => final hidden state
        dV = np.dot(dL_by_doT, h_T.T)
        db_y = dL_by_doT

        # Gradient into final hidden layer from output is scaled by the factor of the weigths in V
        dh = np.dot(self.V.T, dL_by_doT)

        for t in reversed(range(len(sequence) - 1)):
            h_t = h_states[t+1]
            h_prev = h_states[t]

            x_t = sequence[t].reshape(-1, 1)

            # 1 - h_t ** 2 is the derivative of tanh = 1 - (tanh(x)) ** 2
            da = dh * (1 - h_t ** 2)

            # Updating shared parameters in each time step
            dU += np.dot(
                da,
                x_t.T
            )

            dW += np.dot(
                da,
                h_prev.T
            )

            db_h += da

            # Sending the error in this time-step to previous hidden state scaled by the weight into previous step
            dh = np.dot(
                self.W.T,
                da
            )

        self.save_gradients(dU, dW, dV, db_h, db_y)

        self.U -= self.learning_rate * dU
        self.W -= self.learning_rate * dW
        self.V -= self.learning_rate * dV

        self.b_h -= self.learning_rate * db_h
        self.b_y -= self.learning_rate * db_y

        return loss

    def train(self, sequence, epochs=1000):
        for epoch in range(epochs):
            loss = self.backpropagate_through_time(sequence)

            if epoch % 100 == 0:
                print(
                    f"Epoch {epoch}: Loss = {loss:.4f}"
                )

                print(f"Gradients: ", self.get_gradients()[-1])

    def predict(self, sequence, idx_to_word):
        h_prev = self.initialize_hidden()

        for x_t in sequence:

            x_t = x_t.reshape(-1, 1)

            _, h_prev = self.forward(
                x_t,
                h_prev
            )

        h_T = h_prev

        o_T = np.dot(self.V, h_T) + self.b_y

        y_pred = self.softmax(o_T)

        predicted_idx = np.argmax(y_pred)

        predicted_word = idx_to_word[predicted_idx]

        return predicted_word, y_pred

In [52]:
rnn = RNN(
	input_size=vocab_size,
	hidden_size=10,
	output_size=vocab_size,
	learning_rate=0.01
)

rnn.train(
	one_hot_sequence,
	epochs=500
)

Epoch 0: Loss = 2.2064
Gradients:  (array([[-9.66897368e-04, -1.62382698e-03,  8.74835067e-06,
        -7.73577949e-06,  2.08556404e-04, -1.08997073e-04,
        -9.04400557e-06, -2.96819446e-02, -1.97208734e-02],
       [ 2.79062428e-03,  4.02656200e-03,  4.14786198e-05,
        -2.15916185e-05,  9.56590964e-04, -1.92847564e-04,
         8.79874657e-06,  1.47284094e-01, -2.96271128e-02],
       [-4.68086291e-03, -1.12233673e-02, -2.95711479e-05,
         1.96837102e-05, -1.04909032e-03,  1.93058348e-04,
        -1.18767366e-05, -4.46654647e-02,  1.81639999e-02],
       [-1.89498023e-03,  8.10528642e-03, -2.70767099e-05,
         1.30825910e-05, -3.74226671e-04,  6.17078860e-05,
         3.19880706e-05, -2.95212812e-02,  2.49261809e-02],
       [ 4.46180345e-04, -1.38940406e-03,  1.42410102e-05,
        -5.71035807e-06,  2.43763843e-04, -4.04780104e-06,
        -2.16119618e-05, -6.11157087e-02, -1.37871159e-02],
       [-6.82063117e-03,  7.82458111e-03, -4.20199196e-05,
         4.0094

In [53]:
rnn.predict(one_hot_sequence, idx_to_word)

('bank',
 array([[0.96267437],
        [0.00479915],
        [0.00727212],
        [0.00310083],
        [0.00469881],
        [0.00395224],
        [0.00473763],
        [0.00446707],
        [0.00429777]]))